In [3]:
%pip install yfinance

  Using cached yfinance-1.8.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached beautifulsoup4-4.15.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached curl_cffi-0.17.0-cp310-abi3-win_amd64.whl.metadata (19 kB)
  Using cached lxml-6.1.3-cp313-cp313-win_amd64.whl.metadata (3.4 kB)
  Using cached peewee-4.5.3-py3-none-any.whl.metadata (10 kB)
  Using cached cffi-2.1.1-cp313-cp313-win_amd64.whl.metadata (2.6 kB)
  Using cached certifi-2026.10.23-py3-none-any.whl.metadata (2.5 kB)
Using cached yfinance-1.8.0-py3-none-any.whl (158 kB)
Using cached beautifulsoup4-4.15.0-py3-none-any.whl (109 kB)
Using cached curl_cffi-0.17.0-cp310-abi3-win_amd64.whl (2.1 MB)
Using cached certifi-2026.10.23-py3-none-any.whl (135 kB)
Using cached cffi-2.1.1-cp313-cp313-win_amd64.whl (185 kB)
Using cached lxml-6.1.3-cp313-cp313-win_amd64.whl (4.0 MB)
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
    -------------------

In [1]:
import yfinance as yf
import pandas as pd

ticker = "TCS.NS"
stock = yf.Ticker(ticker)
data = stock.history(period="5y", interval="1d")

In [2]:
print(data.shape)
print(data.info())

(1236, 7)
<class 'pandas.DataFrame'>
DatetimeIndex: 1236 entries, 2021-11-22 00:00:00+05:30 to 2026-11-19 00:00:00+05:30
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1236 non-null   float64
 1   High          1236 non-null   float64
 2   Low           1236 non-null   float64
 3   Close         1236 non-null   float64
 4   Volume        1236 non-null   int64  
 5   Dividends     1236 non-null   float64
 6   Stock Splits  1236 non-null   float64
dtypes: float64(6), int64(1)
memory usage: 77.2 KB
None


In [3]:
# Create a copy of the original dataset
df = data.copy()

print(df.head())
print("Dataset shape:", df.shape)

                                  Open         High          Low        Close  \
Date                                                                            
2021-11-22 00:00:00+05:30  3011.469555  3043.978994  2978.484115  2994.154297   
2021-11-23 00:00:00+05:30  2982.553059  3013.677192  2950.346723  2999.218994   
2021-11-24 00:00:00+05:30  3006.101761  3024.153001  2964.372033  2981.081299   
2021-11-25 00:00:00+05:30  2981.600750  2998.136905  2973.073112  2983.332275   
2021-11-26 00:00:00+05:30  2965.237540  3021.512121  2953.895963  2984.154541   

                            Volume  Dividends  Stock Splits  
Date                                                         
2021-11-22 00:00:00+05:30  3167936        0.0           0.0  
2021-11-23 00:00:00+05:30  2153461        0.0           0.0  
2021-11-24 00:00:00+05:30  2161746        0.0           0.0  
2021-11-25 00:00:00+05:30  1849619        0.0           0.0  
2021-11-26 00:00:00+05:30  1941251        0.0           0.0 

In [4]:
# Check missing values in each column
print(df.isnull().sum())

Open            0
High            0
Low             0
Close           0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64


In [5]:
# Count duplicate rows
print("Duplicate rows:", df.duplicated().sum())

# Check duplicate dates
print("Duplicate dates:", df.index.duplicated().sum())

Duplicate rows: 0
Duplicate dates: 0


In [6]:
# Check data types of all columns
print(df.dtypes)

Open            float64
High            float64
Low             float64
Close           float64
Volume            int64
Dividends       float64
Stock Splits    float64
dtype: object


In [7]:
# Check for non-positive stock prices
price_columns = ['Open', 'High', 'Low', 'Close']

print("Non-positive prices:")
print((df[price_columns] <= 0).sum())

# Check negative volume
print("Negative volume:", (df['Volume'] < 0).sum())

Non-positive prices:
Open     0
High     0
Low      0
Close    0
dtype: int64
Negative volume: 0


In [8]:
# Remove exact duplicate rows
df = df.drop_duplicates()

# Remove duplicate dates, keeping the first occurrence
df = df[~df.index.duplicated(keep='first')]

# Remove rows containing missing values
df = df.dropna()

print("Shape after preprocessing:", df.shape)
print("Missing values remaining:")
print(df.isnull().sum())

Shape after preprocessing: (1236, 7)
Missing values remaining:
Open            0
High            0
Low             0
Close           0
Volume          0
Dividends       0
Stock Splits    0
dtype: int64


In [9]:
# Select relevant columns for initial prediction
features = ['Open', 'High', 'Low', 'Close', 'Volume']

processed_data = df[features].copy()

print(processed_data.head())
print("Processed shape:", processed_data.shape)

                                  Open         High          Low        Close  \
Date                                                                            
2021-11-22 00:00:00+05:30  3011.469555  3043.978994  2978.484115  2994.154297   
2021-11-23 00:00:00+05:30  2982.553059  3013.677192  2950.346723  2999.218994   
2021-11-24 00:00:00+05:30  3006.101761  3024.153001  2964.372033  2981.081299   
2021-11-25 00:00:00+05:30  2981.600750  2998.136905  2973.073112  2983.332275   
2021-11-26 00:00:00+05:30  2965.237540  3021.512121  2953.895963  2984.154541   

                            Volume  
Date                                
2021-11-22 00:00:00+05:30  3167936  
2021-11-23 00:00:00+05:30  2153461  
2021-11-24 00:00:00+05:30  2161746  
2021-11-25 00:00:00+05:30  1849619  
2021-11-26 00:00:00+05:30  1941251  
Processed shape: (1236, 5)


In [ ]:
# Save the preprocessed dataset
processed_data.to_csv("TCS_preprocessed.csv")

print("Preprocessed dataset saved successfully!")